# Test the IBM Cloud Code Engine app

A quick smoke test for the app deployed on **IBM Cloud Code Engine** (`application-spam-filter`, region `us-south`). Right now it runs the Code Engine *hello world* sample image, which answers every request with a banner and some of its environment variables.

The notebook checks four things:

1. The app answers with **HTTP 200**.
2. The page is the **hello world** banner.
3. The environment variables say it is the **right app, region and port**.
4. How **fast** it answers. Code Engine scales to zero, so the first call after a quiet period can take a few seconds (a *cold start*).

It uses only the Python standard library: no `pip install` needed.

## 1. Settings

The URL follows the Code Engine pattern `https://<app>.<subdomain>.<domain>`. Change these values to test another app.

In [ ]:
CE_APP = "application-spam-filter"
CE_SUBDOMAIN = "27d9qlagmosq"
CE_DOMAIN = "us-south.codeengine.appdomain.cloud"
EXPECTED_REGION = "us-south"
EXPECTED_PORT = "8080"   # the port Code Engine gives the container

APP_URL = f"https://{CE_APP}.{CE_SUBDOMAIN}.{CE_DOMAIN}/"
TIMEOUT = 60             # seconds; leaves room for a cold start
print(APP_URL)

## 2. Call the app once

In [ ]:
import time
import urllib.request


def call(url, timeout=TIMEOUT):
    """GET the url. Returns (status code, body text, seconds taken)."""
    start = time.perf_counter()
    with urllib.request.urlopen(url, timeout=timeout) as resp:
        body = resp.read().decode("utf-8", errors="replace")
        status = resp.status
    return status, body, time.perf_counter() - start


status, body, seconds = call(APP_URL)
print(f"HTTP {status} in {seconds:.2f} s\n")
print(body)

## 3. Check the answer

The body ends with `NAME=value` lines. We turn them into a dictionary and compare them with what we expect.

In [ ]:
env = {}
for line in body.splitlines():
    name, sep, value = line.partition("=")
    if sep and name.isupper():
        env[name] = value

for name in ["CE_APP", "CE_REGION", "CE_SUBDOMAIN", "K_REVISION", "HOSTNAME", "PORT"]:
    print(f"{name:13} {env.get(name, '(missing)')}")

In [ ]:
checks = {
    "HTTP status is 200": status == 200,
    "Hello world banner is shown": "Hello World" in body,
    "App name matches": env.get("CE_APP") == CE_APP,
    "Region matches": env.get("CE_REGION") == EXPECTED_REGION,
    "Subdomain matches": env.get("CE_SUBDOMAIN") == CE_SUBDOMAIN,
    "Container port matches": env.get("PORT") == EXPECTED_PORT,
}
for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)

assert all(checks.values()), "Some checks failed: see above"
print("\nAll checks passed.")

## 4. Response time

We call the app a few times. If the app had scaled to zero, the first call includes the cold start and is much slower than the rest. `K_REVISION` and `HOSTNAME` show which revision and which instance answered.

In [ ]:
N_CALLS = 5
times = []
for i in range(1, N_CALLS + 1):
    status, body, seconds = call(APP_URL)
    host = next((l.split("=", 1)[1] for l in body.splitlines() if l.startswith("HOSTNAME=")), "?")
    times.append(seconds)
    print(f"call {i}: HTTP {status}  {seconds:.2f} s  instance {host}")

warm = sorted(times)[len(times) // 2]
print(f"\nfirst call {times[0]:.2f} s, median {warm:.2f} s")
if times[0] > 3 * warm and times[0] > 1:
    print("The first call was much slower: probably a cold start.")

## Next step

Once the spam filter image (built from this repository's `Dockerfile`) replaces the hello world sample, the banner check in section 3 will fail: that is expected. The Streamlit app answers `ok` at `/_stcore/health`, so you can test it with `call(APP_URL + "_stcore/health")` instead. Remember to set the Code Engine app's port to `8501`, the port the Streamlit container listens on.